## RAG 

### Experto en responder preguntas para AgroTech

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [2]:
MODEL = "llama3"
DB_NAME = "test_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Configurar los dos objetos clave de LangChain: «retriever» y «llm»

#### Nota al margen sobre la «temperatura»:
- Controla el grado de diversidad de la salida
- Una temperatura de 0 significa que la salida debe ser predecible
- Una temperatura más alta proporciona mayor variedad en las respuestas

Hay quien describe la temperatura como algo parecido a la «creatividad», pero eso no es del todo correcto
- En realidad, controla qué tokens se seleccionan durante la inferencia
- «temperature=0» significa: seleccionar siempre el token con mayor probabilidad
- «temperature=1» suele significar: un token con un 10 % de probabilidad debería elegirse el 10 % de las veces

Nota: una temperatura de 0 no significa que los resultados vayan a ser siempre reproducibles. También es necesario establecer una semilla aleatoria. 

In [8]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(temperature=0, model="llama3")

### Estos objetos de LangChain implementan el método `invoke`

In [5]:
retriever.invoke("¿Quien es Arcadio?")

[Document(id='cc3ac312-32fa-4a2c-8631-7a409c204e7e', metadata={'doc_type': 'empleados', 'source': '..\\knowledge-base\\empleados\\Alejandro Herrera.md'}, page_content='# Alejandro Herrera\n\n## Resumen\n\n-   **Fecha de nacimiento:** 11 de noviembre de 1988\n-   **Puesto:** Responsable de Control de Calidad y Jefe de Empaquetado\n-   **Ubicación:** Gáldar, Gran Canaria (Islas Canarias)\n-   **Salario actual:** 29.500 €\n-   **Metadatos sugeridos para ChromaDB:** ` {"departamento": "operaciones_almacen", "especialidad": "calibrado_dop", "modulo_asociado": "PrecioLLM"}`\n\n## Progresión de carrera en la Cooperativa'),
 Document(id='c5e6c5a4-a60c-4163-beaa-d934ef94bc9c', metadata={'doc_type': 'compañia', 'source': '..\\knowledge-base\\compañia\\resumen.md'}, page_content='## Desglose del Portafolio de Clientes\n\nLos 32 contratos activos de AgroLLM abarcan todo el espectro de la tecnología y la gestión agrícola:'),
 Document(id='9fd83b97-69f6-4c6d-9e67-ec8d8111a325', metadata={'doc_type':

In [9]:
llm.invoke("¿Quién es Arcadio?")

AIMessage(content='Un personaje interesante!\n\nArcadio puede referirse a varios personajes históricos o literarios. Aquí te presento algunos ejemplos:\n\n1. **Arcadio (rey visigodo)**: Fue un rey visigodo que gobernó el Reino Visigodo de Toledo desde 584 hasta su muerte en 603. Es conocido por haber sido uno de los más importantes monarcas visigodos y por haber promovido la conversión al cristianismo.\n2. **Arcadio (personaje literario)**: Es un personaje principal en la novela "La Comedia Humana" del escritor francés Gustave Flaubert, publicada en 1848. Arcadio es el hijo de Madame Bovary y su marido, Charles Bovary.\n3. **Arcadio (mitología)**: En la mitología griega, Arcadio era un rey de Argos que vivió en el siglo XIII a.C. Según la leyenda, fue el padre de Télefo, un héroe troyano.\n\nEn general, el nombre Arcadio se asocia con la antigüedad y la literatura. Si tienes más información o contexto sobre quién es Arcadio que estás buscando, estaré feliz de ayudarte a encontrar más d

In [10]:
SYSTEM_PROMPT_TEMPLATE = """
Eres un asistente experto y amable que representa a la empresa AgroTech.
Estás chateando con un usuario sobre AgroTech.
Utiliza el contexto proporcionado para responder a cualquier pregunta.
Si no sabes la respuesta, dilo.
Contexto:
{context}
"""

In [17]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    print(docs)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [18]:
answer_question("¿Quién es Arcadio?", [])

[Document(id='cc3ac312-32fa-4a2c-8631-7a409c204e7e', metadata={'doc_type': 'empleados', 'source': '..\\knowledge-base\\empleados\\Alejandro Herrera.md'}, page_content='# Alejandro Herrera\n\n## Resumen\n\n-   **Fecha de nacimiento:** 11 de noviembre de 1988\n-   **Puesto:** Responsable de Control de Calidad y Jefe de Empaquetado\n-   **Ubicación:** Gáldar, Gran Canaria (Islas Canarias)\n-   **Salario actual:** 29.500 €\n-   **Metadatos sugeridos para ChromaDB:** ` {"departamento": "operaciones_almacen", "especialidad": "calibrado_dop", "modulo_asociado": "PrecioLLM"}`\n\n## Progresión de carrera en la Cooperativa'), Document(id='c5e6c5a4-a60c-4163-beaa-d934ef94bc9c', metadata={'doc_type': 'compañia', 'source': '..\\knowledge-base\\compañia\\resumen.md'}, page_content='## Desglose del Portafolio de Clientes\n\nLos 32 contratos activos de AgroLLM abarcan todo el espectro de la tecnología y la gestión agrícola:'), Document(id='9fd83b97-69f6-4c6d-9e67-ec8d8111a325', metadata={'source': '..

'Disculpa, pero no hay información disponible sobre alguien llamado Arcadio en el contexto proporcionado. ¿Podrías proporcionar más detalles o contexto sobre quién es Arcadio y qué relación tiene con AgroTech? Estoy aquí para ayudarte.'

In [19]:
gr.ChatInterface(answer_question).launch()

c:\Users\Usuario\Desktop\MasterIA\Asignaturas\Segundo_Cuatrimestre\TFM\Arquitectura-RAG-Multimodal\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


[Document(id='2d52e166-0f20-49a9-8bf2-a511176f5797', metadata={'doc_type': 'empleados', 'source': '..\\knowledge-base\\empleados\\Alejandro Castro.md'}, page_content='# Alejandro Castro\n\n## Resumen\n\n-   **Fecha de nacimiento:** 24 de junio de 1981\n-   **Puesto:** Ingeniero Agrónomo y Director Técnico de Campo\n-   **Ubicación:** Los Llanos de Aridane, La Palma (Islas Canarias)\n-   **Salario actual:** 42.000 €\n-   **Metadatos sugeridos para ChromaDB:** `{"departamento": "tecnico_campo", "especialidad": "musaceas", "modulo_asociado": "FitoLLM"}`\n\n## Progresión de carrera en la Cooperativa'), Document(id='cc3ac312-32fa-4a2c-8631-7a409c204e7e', metadata={'doc_type': 'empleados', 'source': '..\\knowledge-base\\empleados\\Alejandro Herrera.md'}, page_content='# Alejandro Herrera\n\n## Resumen\n\n-   **Fecha de nacimiento:** 11 de noviembre de 1988\n-   **Puesto:** Responsable de Control de Calidad y Jefe de Empaquetado\n-   **Ubicación:** Gáldar, Gran Canaria (Islas Canarias)\n-   *

## Admit it - you thought RAG would be more complicated than that!!